In [ ]:
import numpy as np
import pandas as pd
import scipy

# learners
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.model_selection import train_test_split

# this module
from aipyw import AIPyW
from aipyw.dgp import dgp_discrete, hainmueller

np.random.seed(42)

# Basic Demo


## Discrete Treatments

In [2]:
true_effects = np.array([0.0, 0.4, 0.5, 0.55])
Y, D, X = dgp_discrete(
    n=200_000,
    p=4,
    treat_effects=true_effects,
)
Y.shape, D.shape, X.shape

((200000,), (200000,), (200000, 10))

In [3]:
(Y[D == 1,].mean() - Y[D == 0,].mean(),
  Y[D == 2,].mean() - Y[D == 0,].mean(),
  Y[D == 3,].mean() - Y[D == 0,].mean()
)

(-0.16899053834328948, 0.6362362958627005, 0.7665792763677035)

Naive estimates badly biased.

### init nuisance functions and tune them

In [4]:
m1 = LGBMRegressor(
    verbose=-1,
    n_jobs=-1,
    n_estimators=200,
)

m2 = LGBMClassifier(
    verbose=-1,
    n_jobs=-1,
    n_estimators=200,
)

### standard AIPW

In [5]:
%%time
doubledouble3 = AIPyW(riesz_method="ipw",
                      propensity_model=m2, outcome_model=m1,
                      )
doubledouble3.fit(X, D, Y)
doubledouble3.summary()

CPU times: user 1min 8s, sys: 20.1 s, total: 1min 28s
Wall time: 14.8 s


{'1 vs 0': {'effect': 0.32334, 'se': 0.00826},
 '2 vs 0': {'effect': 0.46282, 'se': 0.00534},
 '3 vs 0': {'effect': 0.50756, 'se': 0.0058},
 '2 vs 1': {'effect': 0.13948, 'se': 0.00866},
 '3 vs 1': {'effect': 0.18423, 'se': 0.00895},
 '3 vs 2': {'effect': 0.04474, 'se': 0.00637}}

Calibrate the nuisance functions per [Van der Laan et al (2024)](https://arxiv.org/pdf/2411.02771v1).

In [6]:
%%time
doubledouble3 = AIPyW(riesz_method="ipw",
                      propensity_model=m2,
                      outcome_model=m1,
                      calibrate = True)
doubledouble3.fit(X, D, Y)
doubledouble3.summary()

CPU times: user 1min 23s, sys: 19 s, total: 1min 42s
Wall time: 16 s


{'1 vs 0': {'effect': 0.41359, 'se': 0.12007},
 '2 vs 0': {'effect': 0.54448, 'se': 0.09774},
 '3 vs 0': {'effect': 0.52969, 'se': 0.10547},
 '2 vs 1': {'effect': 0.13089, 'se': 0.09707},
 '3 vs 1': {'effect': 0.1161, 'se': 0.10484},
 '3 vs 2': {'effect': -0.01479, 'se': 0.07829}}

### hajek AIPW

In [7]:
%%time
doubledouble3 = AIPyW(riesz_method="ipw-hajek")
doubledouble3.fit(X, D, Y)
doubledouble3.summary()

CPU times: user 3.6 s, sys: 5.27 s, total: 8.87 s
Wall time: 600 ms


{'1 vs 0': {'effect': 0.40009, 'se': 1e-05},
 '2 vs 0': {'effect': 0.49897, 'se': 1e-05},
 '3 vs 0': {'effect': 0.55053, 'se': 1e-05},
 '2 vs 1': {'effect': 0.09888, 'se': 1e-05},
 '3 vs 1': {'effect': 0.15043, 'se': 1e-05},
 '3 vs 2': {'effect': 0.05156, 'se': 1e-05}}

### linear riesz representer

In [8]:
%%time
doubledouble3 = AIPyW(riesz_method="linear")
doubledouble3.fit(X, D, Y)
doubledouble3.summary()

CPU times: user 54 s, sys: 16.4 s, total: 1min 10s
Wall time: 5.1 s


{'1 vs 0': {'effect': 0.35072, 'se': 0.00023},
 '2 vs 0': {'effect': 0.43005, 'se': 0.00024},
 '3 vs 0': {'effect': 0.47835, 'se': 0.00023},
 '2 vs 1': {'effect': 0.07933, 'se': 0.00018},
 '3 vs 1': {'effect': 0.12763, 'se': 0.00017},
 '3 vs 2': {'effect': 0.0483, 'se': 0.0002}}

### balancing weights

Compare the three `CovariateBalancer` backends exposed through `AIPyW(riesz_method="balancing")`: quadratic/L2 calibration, entropy balancing, and the Adelie-backed balnet calibration path.

In [ ]:
%%time
BALANCING_OBJS = ["quadratic", "entropy", "balnet"]

balancing_discrete = {}
for bal_obj in BALANCING_OBJS:
    model = AIPyW(riesz_method="balancing", bal_obj=bal_obj)
    model.fit(X, D, Y)
    balancing_discrete[bal_obj] = model.summary()

pd.DataFrame({k: {kk: vv["effect"] for kk, vv in v.items()} for k, v in balancing_discrete.items()})


## Hainmueller (2012) Simulation study

Binary treatment, continuous outcome, 2 groups. We parametrize degree of overlap, functional form of outcome and treatment models. True effect is zero, so RMSE is easy to calculate.

In [ ]:
def one_rep(
    n_samples, overlap_design, pscore_design, outcome_design, riesz_method, **kwargs
):
    # generate data
    y, d, X = hainmueller(
        n_samples=n_samples,
        overlap_design=overlap_design,
        pscore_design=pscore_design,
        outcome_design=outcome_design,
    )
    m1, m2 = LGBMRegressor(verbose=-1, n_jobs=1), LGBMClassifier(verbose=-1, n_jobs=1)
    # model instantiation
    aipw = AIPyW(
        propensity_model=m2, outcome_model=m1, riesz_method=riesz_method, **kwargs
    )
    aipw.fit(X, d, y, n_rff=100)
    return aipw.summary()["1 vs 0"]["effect"]

Favorable case: good overlap, linear pscore and outcome

In [ ]:
%%time
one_rep(10_000, 2, 1, 1, "ipw")

CPU times: user 795 ms, sys: 498 ms, total: 1.29 s
Wall time: 530 ms


-0.00333

In [ ]:
%%time
one_rep(10_000, 2, 1, 1, "ipw-hajek")

CPU times: user 399 ms, sys: 869 µs, total: 400 ms
Wall time: 399 ms


-0.00416

In [ ]:
%%time
one_rep(10_000, 2, 1, 1, "linear")

CPU times: user 4.19 s, sys: 1.6 s, total: 5.79 s
Wall time: 665 ms


0.00587

In [ ]:
%%time
one_rep(10_000, 2, 1, 1, "kernel")

CPU times: user 11.5 s, sys: 3.15 s, total: 14.6 s
Wall time: 1.23 s


0.01437

In [ ]:
%%time
one_rep(10_000, 2, 1, 1, "automatic")

CPU times: user 886 ms, sys: 558 ms, total: 1.44 s
Wall time: 338 ms


0.02806

In [ ]:
%%time
pd.Series(
    {
        bal_obj: one_rep(10_000, 2, 1, 1, "balancing", bal_obj=bal_obj)
        for bal_obj in BALANCING_OBJS
    },
    name="good overlap / linear pscore / linear outcome",
)


In [ ]:
# The previous cell runs all three balancing Riesz variants.
# Kept as a placeholder so old cell references remain stable.


### Hard case: poor overlap, non-linear pscore and outcome

In [ ]:
%%time
one_rep(10_000, 1, 3, 3, "ipw")

CPU times: user 889 ms, sys: 788 ms, total: 1.68 s
Wall time: 472 ms


1.29438

In [ ]:
%%time
one_rep(10_000, 1, 3, 3, "ipw-hajek")

CPU times: user 394 ms, sys: 3.23 ms, total: 397 ms
Wall time: 395 ms


-5.33708

In [ ]:
%%time
one_rep(10_000, 1, 3, 3, "linear")

CPU times: user 2.87 s, sys: 997 ms, total: 3.87 s
Wall time: 557 ms


-2.79707

In [ ]:
%%time
one_rep(10_000, 1, 3, 3, "kernel")

CPU times: user 14.4 s, sys: 2.68 s, total: 17.1 s
Wall time: 1.39 s


-1.08467

In [ ]:
%%time
one_rep(10_000, 2, 1, 1, "automatic")

CPU times: user 881 ms, sys: 635 ms, total: 1.52 s
Wall time: 369 ms


0.02591

In [ ]:
%%time
pd.Series(
    {
        bal_obj: one_rep(10_000, 1, 3, 3, "balancing", bal_obj=bal_obj)
        for bal_obj in BALANCING_OBJS
    },
    name="poor overlap / trig pscore / very nonlinear outcome",
)


In [ ]:
# The previous cell runs all three balancing Riesz variants.
# Kept as a placeholder so old cell references remain stable.


### all together

In [ ]:
from joblib import Parallel, delayed

def compute_ate_rmse_parallel(
    n_samples,
    overlap_design,
    pscore_design,
    outcome_design,
    riesz_method,
    n_replications=100,
    n_jobs=-1,
    **kwargs,
):
    ate_estimates = Parallel(n_jobs=n_jobs)(
        delayed(one_rep)(
            n_samples,
            overlap_design,
            pscore_design,
            outcome_design,
            riesz_method,
            **kwargs,
        )
        for _ in range(n_replications)
    )
    true_ate = 0
    rmse = np.sqrt(np.mean((np.array(ate_estimates) - true_ate) ** 2))
    return rmse


In [ ]:
%%time
from itertools import product

base_specs = [
    ("ipw", "ipw", {}),
    ("ipw-hajek", "ipw-hajek", {}),
    ("linear", "linear", {}),
    ("kernel", "kernel", {}),
    ("automatic", "automatic", {}),
]
balancing_specs = [
    (f"balancing_{bal_obj}", "balancing", {"bal_obj": bal_obj})
    for bal_obj in BALANCING_OBJS
]
est_specs = base_specs + balancing_specs

params = np.arange(1, 4)
param_list = list(product(params, params, params, est_specs))
res_dict = {}
for overlap_design, pscore_design, outcome_design, spec in param_list:
    label, method, kwargs = spec
    key = "_".join([str(overlap_design), str(pscore_design), str(outcome_design), label])
    res_dict[key] = compute_ate_rmse_parallel(
        10_000,
        overlap_design,
        pscore_design,
        outcome_design,
        method,
        **kwargs,
    )


In [ ]:
import pandas as pd

designs = list(
    product(
        ["poor", "good", "medium"],
        ["linear", "quad", "trig"],
        ["linear", "quad", "nl"],
    )
)
res_df = pd.DataFrame({"design": designs})
for label, _, _ in est_specs:
    res_df[label] = [v for k, v in res_dict.items() if k.endswith(label)]

# unpack design column
res_df["overlap_design"] = res_df["design"].apply(lambda x: x[0])
res_df["pscore_design"] = res_df["design"].apply(lambda x: x[1])
res_df["outcome_design"] = res_df["design"].apply(lambda x: x[2])
res_df.drop(columns=["design"], inplace=True)
res_df


## Kang--Schafer-style stress test

A small transformed-covariate design in the spirit of Kang and Schafer (2007). The observed covariates are nonlinear transformations of latent normal confounders, so simple balancing on the observed columns is intentionally strained.

In [ ]:
def kang_schafer_dgp(n=10_000, tau=1.0, seed=None):
    rng = np.random.default_rng(seed)
    Z = rng.normal(size=(n, 4))
    X = np.column_stack(
        [
            np.exp(Z[:, 0] / 2),
            Z[:, 1] / (1 + np.exp(Z[:, 0])) + 10,
            (Z[:, 0] * Z[:, 2] / 25 + 0.6) ** 3,
            (Z[:, 1] + Z[:, 3] + 20) ** 2,
        ]
    )
    propensity = scipy.special.expit(-Z[:, 0] + 0.5 * Z[:, 1] - 0.25 * Z[:, 2] - 0.1 * Z[:, 3])
    W = rng.binomial(1, propensity)
    mu0 = 210 + 27.4 * Z[:, 0] + 13.7 * Z[:, 1] + 13.7 * Z[:, 2] + 13.7 * Z[:, 3]
    Y = mu0 + tau * W + rng.normal(size=n)
    return Y, W, X


def kang_schafer_rep(n_samples, riesz_method, **kwargs):
    y, d, X = kang_schafer_dgp(n=n_samples)
    m1, m2 = LGBMRegressor(verbose=-1, n_jobs=1), LGBMClassifier(verbose=-1, n_jobs=1)
    aipw = AIPyW(propensity_model=m2, outcome_model=m1, riesz_method=riesz_method, **kwargs)
    aipw.fit(X, d, y, n_rff=100)
    return aipw.summary()["1 vs 0"]["effect"]


In [ ]:
%%time
pd.Series(
    {
        bal_obj: kang_schafer_rep(10_000, "balancing", bal_obj=bal_obj)
        for bal_obj in BALANCING_OBJS
    },
    name="Kang-Schafer-style transformed covariates",
)


In [ ]:
%%time

def compute_kang_schafer_rmse(riesz_method, n_samples=5_000, n_replications=100, n_jobs=-1, **kwargs):
    estimates = Parallel(n_jobs=n_jobs)(
        delayed(kang_schafer_rep)(n_samples, riesz_method, **kwargs)
        for _ in range(n_replications)
    )
    return np.sqrt(np.mean((np.array(estimates) - 1.0) ** 2))

kang_schafer_rmse = pd.Series(
    {
        f"balancing_{bal_obj}": compute_kang_schafer_rmse(
            "balancing",
            bal_obj=bal_obj,
        )
        for bal_obj in BALANCING_OBJS
    }
)
kang_schafer_rmse
